# 03 · Hình số đo (đợt 2) — cần detector đã chấm điểm

Ba hình: **A6** audit dữ liệu (chỉ cần manifest), **A2** phân bố điểm,
**A3** ROC/PR. Đợt này cần một lần chạy GPU thật; chưa có thì notebook dừng ở
ô kiểm dữ liệu — đó là hành vi đúng.

Nhãn `R1+R2` (map + detector của mình, ảnh của tác giả). Nếu là `R0` thì đổi
`--label`; nhãn trên hình phải khớp sự thật, không khớp kỳ vọng.

In [ ]:
# Chạy từ thư mục gốc repo. Notebook CHỈ gọi tools/figures.py —
# không tự viết lại logic vẽ, để hình trong notebook và hình trong báo cáo
# luôn là cùng một hình.
import json, os, subprocess, sys
from pathlib import Path

ROOT = Path.cwd()
assert (ROOT / 'tools' / 'figures.py').exists(), 'hãy chạy notebook từ thư mục gốc repo'
FIG = Path(os.environ.get('DIRE_FIG_OUT', 'figs'))

def fig_cmd(*args):
    """Chạy tools/figures.py, in lệnh rồi in đầu ra — lệnh in ra là lệnh tái lập."""
    cmd = [sys.executable, str(ROOT / 'tools' / 'figures.py'), *map(str, args)]
    print('$', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    print(r.stdout.strip() or r.stderr.strip())
    r.check_returncode()
    return r

def sidecar(name):
    """Đọc sidecar: nguồn, n, nhãn. Không có sidecar = hình không kiểm được."""
    return json.loads((FIG / 'data' / f'{name}.json').read_text(encoding='utf-8'))

In [ ]:
SCORES = Path(os.environ.get('DIRE_FIG_SCORES', 'runs/eval/scores.npy'))
LABELS = Path(os.environ.get('DIRE_FIG_LABELS', 'runs/eval/labels.npy'))
GENS = Path(os.environ.get('DIRE_FIG_GENS', 'runs/eval/generators.npy'))
MANIFEST = Path(os.environ.get('DIRE_FIG_MANIFEST', 'data/dire-zero-norm/manifest.tsv'))

for name, p in [('scores', SCORES), ('labels', LABELS), ('generators', GENS), ('manifest', MANIFEST)]:
    print(f'{name:11} {"có" if p.exists() else "CHƯA CÓ":7} {p}')

## A6 — audit dữ liệu (chỉ cần manifest, chạy được sớm nhất)

In [ ]:
if MANIFEST.exists():
    fig_cmd('a6-audit', '--out', FIG, '--manifest', MANIFEST)
    sc = sidecar('a6-audit')
    print('n =', sc['n'], '| thật/giả =', sc['n_real'], '/', sc['n_fake'])
    print('manifest_sha256 (nội dung):', sc['sources']['manifest_sha256'])
else:
    print('chưa có manifest — chạy: python -m dire_zero ingest ...')

## A2 — phân bố điểm theo generator

AUC tính cho từng generator như **thật vs generator đó**, không phải
thật vs phần còn lại. Tập con thuần một lớp cho AUC = NaN, và NaN in ra
thành `—` thì trông như "không có dữ liệu" chứ không phải "code sai".

In [ ]:
if SCORES.exists() and LABELS.exists():
    fig_cmd('a2-scores', '--out', FIG, '--scores', SCORES, '--labels', LABELS,
            *(['--generators', GENS] if GENS.exists() else []), '--label', 'R1+R2')
    print(sidecar('a2-scores')['auc'])
else:
    print('chưa có scores/labels — A2 cần một lần eval thật trên test')

## A3 — ROC/PR

In [ ]:
if SCORES.exists() and LABELS.exists():
    fig_cmd('a3-roc', '--out', FIG, '--scores', SCORES, '--labels', LABELS,
            *(['--generators', GENS] if GENS.exists() else []), '--label', 'R1+R2')
else:
    print('bỏ qua')

## Nhắc lại một điều dễ quên

Ngưỡng phải chọn trên **val**, không trên test. Hình A2/A3 vẽ phân bố trên test
chỉ để *xem*, không để chọn ngưỡng — chọn rồi báo cáo trên cùng tập là tự lừa
mình, và không có gì trong hình cho người đọc biết điều đó.